In [8]:
import requests
import random
import os
import json
import csv
import logging
import numpy as np
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

logging.basicConfig(level=logging.INFO)

# Wikidata

In [3]:
# Decreasing the size of the Wikidata5M dataset

file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_30k.tsv'
num_samples = 40_000

with open(file, 'r') as f:
    lines = f.readlines()

random_sample = random.sample(lines, num_samples)

with open(output_file, 'w') as f:
    f.writelines(random_sample)

## Exploratory Analysis

In [21]:
dataset = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

df = pd.read_csv(dataset, sep='\t', header=None)
df.columns = ['head', 'relation', 'tail']
num_relations = df['relation'].unique()
print(f"Number of relations: {len(num_relations)}")

relations_counts = df['relation'].value_counts()
print(f"Value counts of relation: {relations_counts}")

# choosing relations with more than 50 entities 
relations = relations_counts[relations_counts > 50].index.tolist()
print(f"Number of relations after filtering: {len(relations)}")

Number of relations: 50
Value counts of relation: relation
P31      8392
P17      3005
P27      2507
P106     2406
P131     2005
P54      2005
P19      1867
P735     1832
P161     1113
P641     1062
P69       960
P47       906
P421      882
P105      817
P136      806
P171      750
P20       609
P495      569
P1412     521
P1344     486
P166      398
P175      396
P413      396
P264      334
P155      320
P156      305
P364      297
P361      295
P102      276
P734      235
P150      235
P57       215
P463      210
P279      208
P407      205
P159      180
P108      175
P39       165
P3373     164
P937      156
P607      153
P360      144
P527      143
P40       140
P86       136
P162      128
P50       125
P137      122
P141      113
P22       108
Name: count, dtype: int64
Number of relations after filtering: 50


In [24]:
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'

# Initialize a dictionary to count triples for each relation
relation_counts = defaultdict(int)

# Load and analyze the dataset
with open(file_path, 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) >= 3:
            # Assuming the format is: head_entity, relation, tail_entity
            relation = parts[1]
            relation_counts[relation] += 1

# Sort the relations by their counts in descending order
sorted_relations = sorted(relation_counts.items(), key=lambda item: item[1], reverse=True)

# Print the total number of relations and some examples of counts
print(f"Total number of unique relations: {len(relation_counts)}")
for relation, count in sorted_relations[:10]:
    print(f"Relation: {relation}, Count: {count}")

Total number of unique relations: 200
Relation: P31, Count: 7547
Relation: P17, Count: 2702
Relation: P27, Count: 2254
Relation: P106, Count: 2164
Relation: P131, Count: 1803
Relation: P54, Count: 1803
Relation: P19, Count: 1679
Relation: P735, Count: 1648
Relation: P161, Count: 1001
Relation: P641, Count: 955


## Reducing the dataset size through proportional sampling

In [23]:
def proportional_sample(file_path, output_file, num_samples, target_num_relations, min_samples_per_relation=10):
    relation_counts = defaultdict(int)
    relation_triples = defaultdict(list)

    # First pass: count occurrences and collect triples
    with open(file_path, 'r') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) >= 3:
                relation = parts[1]
                relation_counts[relation] += 1
                relation_triples[relation].append(line)
    
    # Select the top N relations by count
    top_relations = sorted(relation_counts, key=relation_counts.get, reverse=True)[:target_num_relations]

    # Adjusted total count to only consider top relations
    total_count = sum(relation_counts[relation] for relation in top_relations)
    
    # Proportional sampling within the top relations
    sampled_triples = []
    for relation in top_relations:
        proportion = relation_counts[relation] / total_count
        samples_for_relation = max(int(proportion * num_samples), min_samples_per_relation)
        
        # Ensure not to exceed the actual number of available triples
        samples_for_relation = min(samples_for_relation, len(relation_triples[relation]))
        
        sampled_triples.extend(random.sample(relation_triples[relation], samples_for_relation))

    # Save the reduced and proportionally sampled dataset
    with open(output_file, 'w') as f:
        f.writelines(sampled_triples)

# Example usage parameters
file_path = './../../data/wikidata5m_inductive/wikidata5m_inductive_train.txt'  # Update this to your actual file path
output_file = './../../data/wikidata5m_inductive/wikidata5m_inductive_train_40k.tsv'   # Desired output file path
num_samples = 40000  # Target number of samples in the reduced dataset
target_num_relations = 200  # Target number of relations to keep
min_samples_per_relation = 50  # Minimum samples per relation to ensure representation

# Execute the sampling
proportional_sample(file_path, output_file, num_samples, target_num_relations, min_samples_per_relation)


## Getting Entity and Relation Information

In [ ]:
# Making it faster

def fetch_entity_info(wikidata_ids):
    wikidata_api_url = "https://www.wikidata.org/w/api.php?action=wbgetentities&format=json&ids=" + "|".join(wikidata_ids)
    try:
        response = requests.get(wikidata_api_url)
        response.raise_for_status()
        data = response.json()
        return data['entities']
    except Exception as e:
        print(f"Error fetching entity information: {e}")
        return {}

def get_entity_info(wikidata_ids):
    entities_info = fetch_entity_info(wikidata_ids)
    return entities_info

def add_entity_values_batch(entries, batch_size=50):
    # Split entries into batches
    batches = [entries[i:i+batch_size] for i in range(0, len(entries), batch_size)]

    results = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        for batch in tqdm(batches[:5], desc="Processing batches"):
            futures = []
            for entry in batch:
                wikidata_ids = [entry['sub_id'], entry['pred_id'], entry['obj_id']]
                futures.append(executor.submit(get_entity_info, wikidata_ids))
            for future in futures:
                result = future.result()
                results.append(result)

    return results

def txt_to_jsonl(input_file, output_file):
    with open(input_file, 'r', encoding='utf-8') as txt_file:
        lines = txt_file.readlines()

    jsonl_entries = []
    for line in lines:
        parts = line.strip().split('\t')
        # Extract subject, predicate, and object
        sub_id, pred_id, obj_id = parts
        entry = {
            'sub_id': sub_id,
            'pred_id': pred_id,
            'obj_id': obj_id
        }
        jsonl_entries.append(entry)

    # Batch processing of entries
    results = add_entity_values_batch(jsonl_entries)

    # Combine results and write to JSONL file
    with open(output_file, 'w', encoding='utf-8') as jsonl_file:
        for entry, entities_info in zip(jsonl_entries, results):
            sub_info = entities_info.get(entry['sub_id'], {})
            pred_info = entities_info.get(entry['pred_id'], {})
            obj_info = entities_info.get(entry['obj_id'], {})

            entry['sub_value'] = sub_info.get('labels', {}).get('en', {}).get('value')
            entry['pred_value'] = pred_info.get('labels', {}).get('en', {}).get('value')
            entry['obj_value'] = obj_info.get('labels', {}).get('en', {}).get('value')

            jsonl_file.write(json.dumps(entry, ensure_ascii=False) + '\n')


input_file = '../data/wikidata5m_inductive/wikidata5m_inductive_train_20k.tsv'
output_file = '../data/wikidata5m_inductive/wikidata5m_inductive_train_20k.jsonl'
txt_to_jsonl(input_file, output_file)